In [1]:
!pip install pyswip

In [2]:
import os

# Install SWI-Prolog
if not os.path.exists('/usr/bin/swipl'):
    !sudo apt-get update
    !sudo apt-get install swi-prolog -y

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,929 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,852 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 https:/

In [3]:
from pyswip import Prolog

prolog = Prolog()
prolog.assertz("reduce('Ibuprofen', 'COX-2_enzyme')")
prolog.assertz("cause('COX-2_enzyme', 'prostaglandins')")
prolog.assertz("cause('prostaglandins', 'inflammation')")
prolog.assertz("treat(W, Z) :- reduce(W, X), cause(X, Y), cause(Y, Z)")

# Query it
results = list(prolog.query("treat('Ibuprofen', Who)"))
print(results)

[{'Who': 'inflammation'}]


In [4]:
prolog.assertz("affects('Ibuprofen', 'COX-2', inhibits)")
prolog.assertz("affects('COX-2', 'prostaglandins', activates)")
prolog.assertz("affects('prostaglandins', 'inflammation', activates)")

prolog.assertz("affects('ChemicalA', 'EnzymeB', activates)")
prolog.assertz("affects('EnzymeB', 'ProblemC', activates)")

# Sign composition rules
prolog.assertz("compose(activates, activates, activates)")
prolog.assertz("compose(inhibits, activates, inhibits)")
prolog.assertz("compose(activates, inhibits, inhibits)")
prolog.assertz("compose(inhibits, inhibits, activates)")

# Base case: direct relationship
prolog.assertz("net_effect(X, Z, S) :- affects(X, Z, S)")

# Recursive case: chain through intermediate Y
prolog.assertz("""
    net_effect(X, Z, S) :-
        affects(X, Y, S1),
        net_effect(Y, Z, S2),
        compose(S1, S2, S)
""")

# Treatment rule: X treats Z if X has net inhibitory effect on Z
prolog.assertz("treats(X, Z) :- net_effect(X, Z, inhibits)")

In [5]:
# Should find Ibuprofen treats inflammation
print(list(prolog.query("treats('Ibuprofen', Who)")))

# Should NOT find ChemicalA treats ProblemC
print(list(prolog.query("treats('ChemicalA', Who)")))

# You can also ask: what is the net effect of ChemicalA on ProblemC?
print(list(prolog.query("net_effect('ChemicalA', 'ProblemC', S)")))
# Should return: activates — meaning A makes C worse, not better

[{'Who': 'COX-2'}, {'Who': 'prostaglandins'}, {'Who': 'inflammation'}]
[]
[{'S': 'activates'}]
